In [2]:
import os
import zipfile
import re
import json

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
# Update this with the path to your downloaded Juliet suite zip file
zip_file_path = os.path.expanduser('C:/Users/Jennifer_Nishimura/Documents/DATASCI266/final_project/W266-Final-Project/Juliet_Test_Suite_v1.3_for_C_Cpp.zip') 
extraction_target_dir = "./extracted_juliet_suite"
output_manifest_file = "juliet_mining_pairs.json"

# Check if file exists (Your custom validation safety-gate)
if not os.path.exists(zip_file_path):
    print(f"Error: Zip file not found at {zip_file_path}")
    print("Please provide the path to your zip file:")
    print(f"  - Place it in: {zip_file_path}")
    print(f"  - Or update the zip_file_path variable above")
    raise FileNotFoundError(f"Zip file not found: {zip_file_path}")

print(f"[+] Found zip file: {zip_file_path}")

# ==========================================
# 2. AUTOMATED EXTRACTION STEP
# ==========================================
print(f"[+] Extracting files to: {extraction_target_dir} ...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_target_dir)
print("[+] Extraction complete.")

# ==========================================
# 3. PROCESSING & SEPARATION FUNCTIONS
# ==========================================
def scan_and_pair_extracted_files(base_directory):
    """
    Recursively searches through all unzipped subdirectories 
    to track down related Juliet source patterns.
    """
    file_groups = {}
    
    # os.walk scans deep subdirectories (CWE121, CWE124, etc.) automatically
    for root, dirs, files in os.walk(base_directory):
        for filename in files:
            if filename.endswith(('.cpp', '.c', '.h')):
                # Capture the shared signature file prefix safely
                match = re.match(r"(CWE\d+.*?)(_(bad|goodG2B|goodB2G|goodG2B1|goodG2B2))?\.(cpp|c|h)$", filename)
                if match:
                    base_name = match.group(1)
                    if base_name not in file_groups:
                        file_groups[base_name] = {"bad": None, "goods": []}
                    
                    full_path = os.path.join(root, filename)
                    
                    if "_bad" in filename:
                        file_groups[base_name]["bad"] = full_path
                    elif "_good" in filename:
                        file_groups[base_name]["goods"].append(full_path)
                        
    return file_groups

def extract_function_body(file_path):
    """Safely extracts text logs from target source pathways."""
    if not file_path or not os.path.exists(file_path):
        return ""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

# ==========================================
# 4. EXECUTION MATRIX PIPELINE
# ==========================================
print("[+] Mapping Juliet folder structures into pairings...")
file_pairs = scan_and_pair_extracted_files(extraction_target_dir)
manifest = []

for base_name, paths in file_pairs.items():
    # Only pair if we have both an exploit file and a matching remediation block
    if paths["bad"] and paths["goods"]:
        bad_code_context = extract_function_body(paths["bad"])
        
        for good_path in paths["goods"]:
            good_code_truth = extract_function_body(good_path)
            
            manifest.append({
                "case_identifier": base_name,
                "bad_file_source": paths["bad"],
                "good_file_source": good_path,
                "prompt_context": bad_code_context,
                "ground_truth_remediation": good_code_truth
            })

# Save output data configurations back to system space
with open(output_manifest_file, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4)

print(f"\n[+] SUCCESS! Processed dataset entries: {len(manifest)}")
print(f"[+] Dataset pairings file written safely to: {output_manifest_file}")


[+] Found zip file: C:/Users/Jennifer_Nishimura/Documents/DATASCI266/final_project/W266-Final-Project/Juliet_Test_Suite_v1.3_for_C_Cpp.zip
[+] Extracting files to: ./extracted_juliet_suite ...
[+] Extraction complete.
[+] Mapping Juliet folder structures into pairings...

[+] SUCCESS! Processed dataset entries: 5868
[+] Dataset pairings file written safely to: juliet_mining_pairs.json


In [ ]:
import json
import os
import random

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
INPUT_MANIFEST = "juliet_mining_pairs.json"
OUTPUT_1K_MANIFEST = "C:/Users/Jennifer_Nishimura/Documents/DATASCI266/final_project/W266-Final-Project/juliet_mining_pairs_5k.json"
SAMPLE_SIZE = 5868

# Safety check for input source files
if not os.path.exists(INPUT_MANIFEST):
    raise FileNotFoundError(
        f"Missing {INPUT_MANIFEST}. Please run your extraction/separation script first!"
    )

# ==========================================
# 2. SAMPLING PIPELINE
# ==========================================
def downsample_manifest():
    print(f"[+] Loading raw manifest: {INPUT_MANIFEST}...")
    with open(INPUT_MANIFEST, 'r', encoding='utf-8') as f:
        master_data = json.load(f)
        
    total_available = len(master_data)
    print(f"[+] Found {total_available} total available case pairings.")
    
    if total_available < SAMPLE_SIZE:
        print(f"[!] Warning: Available cases ({total_available}) are less than the requested sample size ({SAMPLE_SIZE}).")
        print("[!] Copying all available files instead of downsampling.")
        sampled_data = master_data
    else:
        # Using a fixed seed ensures your dataset stays consistent if you rerun the script
        random.seed(42)
        
        # Shuffle the entries to mix various CWE folders and data flows
        print("[+] Shuffling manifest to mix vulnerability categories...")
        random.shuffle(master_data)
        
        # Extract exactly 1000 cases
        sampled_data = master_data[:SAMPLE_SIZE]
        
    # Write out the downsampled subset back to disk
    with open(OUTPUT_1K_MANIFEST, 'w', encoding='utf-8') as f:
        json.dump(sampled_data, f, indent=4)
        
    print(f"\n[+] SUCCESS! Sampled exactly {len(sampled_data)} records.")
    print(f"[+] 1K Downsampled manifest written safely to: {OUTPUT_1K_MANIFEST}")

if __name__ == "__main__":
    downsample_manifest()

[+] Loading raw manifest: juliet_mining_pairs.json...
[+] Found 5868 total available case pairings.
[+] Shuffling manifest to mix vulnerability categories...

[+] SUCCESS! Sampled exactly 5868 records.
[+] 1K Downsampled manifest written safely to: C:/Users/Jennifer_Nishimura/Documents/DATASCI266/final_project/W266-Final-Project/juliet_mining_pairs_10k.json


In [ ]:
import os
import json
import re
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from tree_sitter import Language, Parser
import tree_sitter_cpp as tscpp

# ==========================================
# 1. CONFIGURATION
# ==========================================
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Prevent Rust tokenizer deadlocks


INPUT_1K_MANIFEST = "juliet_mining_pairs_5k.json"
OUTPUT_1K_DATASET = "juliet_hallucination_dataset_5k.json"
MINING_TEMPERATURE = 1.25 # High temp to encourage hallucination
BATCH_SIZE = 8  

# Standard library / STL surface we consider "known", not hallucinated.
# (string.h, stdio.h, cstring, <string>, <vector>, <memory>, etc).
VALID_STANDARD_APIS = {
    # libc string/mem
    "strcpy", "strncpy", "strcat", "strncat", "strlen", "strcmp", "strncmp",
    "memcpy", "memmove", "memset", "memcmp", "strdup", "sprintf", "snprintf",
    "sscanf", "malloc", "calloc", "realloc", "free", "atoi", "atol", "atof",
    # libc stdio
    "printf", "fprintf", "puts", "fputs", "fopen", "fclose", "fread", "fwrite",
    "fgets", "gets", "scanf",
    # C++ STL containers / string
    "push_back", "pop_back", "emplace_back", "size", "length", "empty",
    "clear", "resize", "reserve", "at", "insert", "erase", "find", "substr",
    "c_str", "data", "begin", "end", "front", "back",
    # smart pointers / casts
    "make_unique", "make_shared", "reset", "release", "static_cast",
    "dynamic_cast", "const_cast", "reinterpret_cast",
    # streams
    "cout", "cerr", "cin", "endl", "getline",
    # <cmath> / <cstdlib> — previously missing, caused false PHANTOM_API_OR_VARIABLE
    # flags on legitimate code (e.g. fabs() in a divide-by-zero remediation).
    "fabs", "abs", "labs", "sqrt", "pow", "ceil", "floor", "round",
    "log", "log2", "log10", "exp", "fmod", "rand", "srand", "rand_r",
    # process / system calls commonly used in Juliet-style remediations
    "exit", "system", "getenv", "setenv", "execl", "execv", "execvp",
    "spawnl", "_spawnl", "_wspawnl", "spawnlp", "_spawnlp", "_wspawnlp",
    # wide-char (<cwchar>) equivalents of the narrow-char functions above --
    # missing entirely before, despite wchar_t being common in this dataset.
    "wcscpy", "wcsncpy", "wcscat", "wcsncat", "wcslen", "wcscmp", "wcsncmp",
    "wcschr", "wcsdup", "wmemset", "wmemcpy", "wmemmove", "wmemcmp",
    "swprintf", "fwprintf", "fgetws", "wcstombs", "mbstowcs", "wsprintf",
    # std:: exception types and related calls -- confirmed real via dataset
    # audit (runtime_error/invalid_argument/what were being flagged as
    # phantom despite being completely standard exception-handling code).
    "runtime_error", "invalid_argument", "logic_error", "out_of_range",
    "length_error", "domain_error", "range_error", "overflow_error",
    "underflow_error", "bad_alloc", "bad_cast", "bad_function_call", "what",
    # additional real stdlib/POSIX/Win32 calls confirmed via dataset audit
    "stoi", "stol", "stoul", "stod", "stof", "feof", "ferror", "is_open",
    "flush", "fill", "min", "max", "strnlen", "LoadLibrary", "LoadLibraryEx",
    "LoadLibraryW", "GetLastError", "close", "string", "wstring", "append",
    "assign", "copy", "ignore", "str", "rdbuf", "get", "strlcpy", "wcsstr",
    "wcstoull", "wcstoul", "wcstol", "wcstod", "move", "remove_if", "isprint",
    "to_string", "strcspn", "abort", "write", "fail",
    "SetEnvironmentVariable", "WideCharToMultiByte", "MultiByteToWideChar",
    # OpenLDAP / WinLDAP client functions -- used correctly throughout the
    # LDAP-injection CWE rows in this dataset.
    "ldap_search_ext_s", "ldap_search_ext_sA", "ldap_search_ext_sW",
    "ldap_init", "ldap_initA", "ldap_initW", "ldap_connect", "ldap_unbind",
    "ldap_msgfree",
    # <iomanip> stream manipulators
    "setw", "setfill", "setprecision",
    # Microsoft "safe" (Annex-K-style) CRT functions -- extremely common in
    # exactly this dataset's "here is the secure remediation" code, and were
    # entirely missing before.
    "strcpy_s", "strcat_s", "strncpy_s", "strncat_s", "sprintf_s",
    "sscanf_s", "scanf_s", "memcpy_s", "fopen_s", "wcscpy_s", "wcscat_s",
    "wcsncpy_s", "wcsncat_s", "_wexecv", "_wgetenv",
}

# ==========================================
# 2. AST VALIDATION WORKER
# ==========================================
def worker_verify_ast(llm_code, prompt_context):
    """
    Parses the generated code with tree-sitter and cross-references called
    functions against a standard-library allowlist plus any function names
    that appear in the original prompt context (locally defined APIs).
    """
    cpp_lang = Language(tscpp.language())
    parser = Parser(cpp_lang)

    opens = list(re.finditer(r"```(?:cpp|c\+\+|cxx|c)\s*\n", llm_code))
    if opens:
        start = opens[-1].end()
        close_match = re.search(r"\n```\s*(?:\n|$)", llm_code[start:])
        clean_code = (llm_code[start:start + close_match.start()] if close_match else llm_code[start:]).strip()
    else:
        any_blocks = re.findall(r"```[a-zA-Z]*\s*\n?(.*?)(?:```|\Z)", llm_code, re.DOTALL)
        clean_code = max(any_blocks, key=len).strip() if any_blocks else llm_code.strip()

    diagnostics = {
        "extracted_chars": len(clean_code),
        "brace_balance": clean_code.count("{") - clean_code.count("}"),
        "paren_balance": clean_code.count("(") - clean_code.count(")"),
        "ifdef_count": len(re.findall(r"#\s*if(?:def|ndef)?\b", clean_code)),
        "endif_count": len(re.findall(r"#\s*endif\b", clean_code)),
    }

    if not clean_code:
        return {
            "is_hallucinated": True,
            "hallucination_type": "EMPTY_EXTRACTION",
            "extracted_entity": "N/A",
            "feedback": "No code could be extracted from the model output (no fenced block, or fence contained no text).",
            "extracted_code": "",
            "diagnostics": diagnostics,
        }

    tree = parser.parse(bytes(clean_code, "utf8"))
    root_node = tree.root_node

    if root_node.has_error:
        return {
            "is_hallucinated": True,
            "hallucination_type": "SYNTAX_BREAKDOWN",
            "extracted_entity": "N/A",
            "feedback": "AST validation failed: source code contains broken tokens or unclosed scopes.",
            "extracted_code": clean_code,
            "diagnostics": diagnostics,
        }


    valid_local_apis = set(re.findall(r"\b(\w+)\s*\(", prompt_context))


    locally_defined_functions = set(re.findall(
        r"\b(\w+)\s*\([^()]*\)\s*(?:const\s*)?(?:noexcept\s*)?\{", clean_code
    ))
    valid_local_apis |= locally_defined_functions

    detected_calls = set()


    valid_local_fields = set(re.findall(r"(?:->|\.)\s*(\w+)", prompt_context))
    valid_local_fields |= set(re.findall(r"::\s*~?(\w+)\s*\(", prompt_context))
    detected_fields = set()

    def traverse_nodes(node):
        if node.type == "call_expression":
            function_node = node.child_by_field_name("function")
            if function_node:
                func_name = clean_code[function_node.start_byte:function_node.end_byte].strip()
                func_name = func_name.split("::")[-1]
                func_name = func_name.split(".")[-1].split("->")[-1]
                func_name = func_name.split("<")[0].strip()
                if not func_name.startswith("~"):
                    detected_calls.add(func_name)
        elif node.type == "field_expression":
            field_node = node.child_by_field_name("field")
            if field_node:
                field_name = clean_code[field_node.start_byte:field_node.end_byte].strip()
                detected_fields.add(field_name)
        for child in node.children:
            traverse_nodes(child)

    traverse_nodes(root_node)

    phantom_apis = [
        call for call in detected_calls
        if call and call not in VALID_STANDARD_APIS and call not in valid_local_apis
        and call not in ("get", "cin", "main", "static_cast") and not call.isnumeric()
        and call not in (
            "void", "int", "char", "wchar_t", "bool", "float", "double",
            "long", "short", "unsigned", "signed", "size_t", "auto",
        )
        and re.fullmatch(r"\w+", call)
    ]
    phantom_fields = [
        f for f in detected_fields
        if f and f not in valid_local_fields and f not in VALID_STANDARD_APIS
        and f not in locally_defined_functions
    ]

    if phantom_apis:
        return {
            "is_hallucinated": True,
            "hallucination_type": "PHANTOM_API_OR_VARIABLE",
            "extracted_entity": phantom_apis,
            "feedback": f"AST uncovered hallucinated APIs: {phantom_apis}",
            "extracted_code": clean_code,
            "diagnostics": diagnostics,
        }

    if phantom_fields:
        return {
            "is_hallucinated": True,
            "hallucination_type": "PHANTOM_FIELD_ACCESS",
            "extracted_entity": phantom_fields,
            "feedback": f"AST uncovered member/field access not present in the original prompt: {phantom_fields}",
            "extracted_code": clean_code,
            "diagnostics": diagnostics,
        }

    return {
        "is_hallucinated": False,
        "hallucination_type": "NONE",
        "extracted_entity": "N/A",
        "feedback": "AST parsed successfully with zero syntax errors, phantom APIs, or phantom field accesses.",
        "extracted_code": clean_code,
        "diagnostics": diagnostics,
    }


# ==========================================
# 3. BATCHED GENERATION FUNCTION
# ==========================================
def score_generation(raw_generated_text, prompt_context):
    """Runs the duplicate check + AST validation for one generated sample."""
    prompt_stripped = "".join(prompt_context.split())
    gen_stripped = "".join(raw_generated_text.split())
    is_duplicate = (
        prompt_stripped in gen_stripped
        or gen_stripped in prompt_stripped
        or len(raw_generated_text) < 10
    )

    if is_duplicate:

        validation = {
            "is_hallucinated": True,
            "hallucination_type": "DUPLICATE_OR_DEGENERATE",
            "extracted_entity": "N/A",
            "feedback": "Generation failure: model copied prompt verbatim or hit a repetitive loop constraint.",
            "extracted_code": "",
            "diagnostics": {},
        }
    else:
        try:
            validation = worker_verify_ast(raw_generated_text, prompt_context)
        except Exception as e:
            validation = {
                "is_hallucinated": True,
                "hallucination_type": "AST_PARSE_ERROR",
                "extracted_entity": "N/A",
                "feedback": f"AST validation raised an exception: {e}",
                "extracted_code": "",
                "diagnostics": {},
            }

    return validation, is_duplicate


def generate_batch(rows, model, tokenizer, system_instruction):
    """
    Runs one batched generation + validation pass over a list of manifest rows.

    Uses left-padding so that every sequence in the batch ends at the same
    position; that lets us slice all generated continuations at the same
    fixed offset (the padded prompt length) instead of tracking a different
    prompt length per row.
    """
    prompt_contexts = [row["prompt_context"] for row in rows]
    texts = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": f"Remediate this vulnerable code block securely:\n\n{pc}"},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for pc in prompt_contexts
    ]

    try:
        model_inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True, max_length=4096
        ).to(model.device)
        input_len = model_inputs.input_ids.shape[1]

        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=1024,
                temperature=MINING_TEMPERATURE,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
            )


        pure_output_ids = generated_ids[:, input_len:]
        raw_texts = tokenizer.batch_decode(pure_output_ids, skip_special_tokens=True)
        raw_texts = [t.strip() for t in raw_texts]

        del model_inputs, generated_ids, pure_output_ids

    except Exception as e:

        error_validation = {
            "is_hallucinated": True,
            "hallucination_type": "GENERATION_ERROR",
            "extracted_entity": "N/A",
            "feedback": f"Batched generation raised an exception: {e}",
            "extracted_code": "",
            "diagnostics": {},
        }
        return [("", error_validation, False) for _ in rows]

    results = []
    for raw_generated_text, prompt_context in zip(raw_texts, prompt_contexts):
        validation, is_duplicate = score_generation(raw_generated_text, prompt_context)
        results.append((raw_generated_text, validation, is_duplicate))

    return results


# ==========================================
# 4. MAIN
# ==========================================
def main():
    if not os.path.exists(INPUT_1K_MANIFEST):
        raise FileNotFoundError(f"Missing {INPUT_1K_MANIFEST}. Run your sampling script first!")

    with open(INPUT_1K_MANIFEST, "r", encoding="utf-8") as f:
        manifest_data = json.load(f)

    print("[+] Loading model and tokenizer...")
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
    model.eval()

    system_instruction = (
        "You are a strict C++ refactoring tool. "
        "Fix the security vulnerability in the provided code block. "
        "CRITICAL: Do not repeat any comment headers from the prompt. Do not write a main function. "
        "Output ONLY the corrected class namespace and function implementation wrapper inside markdown."
    )

    print(f"[*] Generating for {len(manifest_data)} rows in batches of {BATCH_SIZE}...")
    results = []
    total_batches = (len(manifest_data) + BATCH_SIZE - 1) // BATCH_SIZE
    for i, start in enumerate(tqdm(range(0, len(manifest_data), BATCH_SIZE))):
        batch_rows = manifest_data[start:start + BATCH_SIZE]
        batch_results = generate_batch(batch_rows, model, tokenizer, system_instruction)
        results.extend(batch_results)

        n_flagged = sum(r[1]["is_hallucinated"] for r in batch_results)
        print(f"[batch {i + 1}/{total_batches}] {len(batch_rows)} rows, {n_flagged} flagged hallucinated")

    dataset_records = []
    for row, (raw_generated_text, validation, is_duplicate) in zip(manifest_data, results):
        dataset_records.append({
            "case_identifier": row["case_identifier"],
            "prompt_context": row["prompt_context"],
            "ground_truth_remediation": row["ground_truth_remediation"],
            "model_generated_output": raw_generated_text,
            "extracted_code": validation.get("extracted_code", ""),
            "is_hallucinated": validation["is_hallucinated"],
            "hallucination_type": validation["hallucination_type"],
            "hallucinated_entity": validation["extracted_entity"],
            "compiler_raw_log": validation["feedback"],
            "diagnostics": validation.get("diagnostics", {}),
        })

    with open(OUTPUT_1K_DATASET, "w", encoding="utf-8") as f:
        json.dump(dataset_records, f, indent=4)

    n_hallucinated = sum(r["is_hallucinated"] for r in dataset_records)
    print(f"\n[+] Done. {n_hallucinated}/{len(dataset_records)} rows flagged as hallucinated.")

    type_counts = {}
    for r in dataset_records:
        type_counts[r["hallucination_type"]] = type_counts.get(r["hallucination_type"], 0) + 1
    print("[+] Breakdown by hallucination_type:")
    for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"    {t}: {c}")

    print(f"[+] Output stored at: {OUTPUT_1K_DATASET}")


if __name__ == "__main__":
    main()

[+] Loading model and tokenizer...


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 89.54it/s] 


[*] Generating for 5868 rows in batches of 8...


  0%|          | 1/734 [00:30<6:14:33, 30.66s/it]

[batch 1/734] 8 rows, 5 flagged hallucinated


...

100%|█████████▉| 733/734 [4:16:56<00:15, 15.99s/it]

[batch 733/734] 8 rows, 4 flagged hallucinated


100%|██████████| 734/734 [4:17:09<00:00, 21.02s/it]

[batch 734/734] 4 rows, 2 flagged hallucinated



[+] Done. 3101/5868 rows flagged as hallucinated.
[+] Breakdown by hallucination_type:
    NONE: 2767
    SYNTAX_BREAKDOWN: 1909
    PHANTOM_API_OR_VARIABLE: 662
    EMPTY_EXTRACTION: 329
    DUPLICATE_OR_DEGENERATE: 148
    PHANTOM_FIELD_ACCESS: 53
[+] Output stored at: juliet_hallucination_dataset_10k.json


OSError: [Errno 22] Invalid argument: '--f=c:\\Users\\Jennifer_Nishimura\\AppData\\Roaming\\jupyter\\runtime\\kernel-v342f0cc7aa05ca919c5e8585e6899688d01edecc3.json'